# 20.18 — Evaluation harnesses

An evaluation harness is a repeatable measurement machine: fixed cases, metrics, slices, uncertainty, and release thresholds.

Experiment tracking supplies run identity, and monitoring tells which slices and failures the harness must cover. Harnesses govern LLMOps, CI/CD, and model registry promotion. Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 17
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)

## The concept, built once

Harness score and pairwise win rate: $\hat m=\frac{1}{n}\sum_i s_i$ and $\text{win}=w/(w+l)$.

The reusable harness scores cases, keeps slice metrics, estimates uncertainty, and returns a release decision rather than a loose number.

In [ ]:
def eval_harness(scores, slices, threshold=0.86, slice_threshold=0.82, rng=None, bootstrap_rounds=200):
    score_array = np.asarray(scores, dtype=float)
    slice_array = np.asarray(slices)
    mean_score = float(np.mean(score_array))
    slice_scores = {}
    for name in np.unique(slice_array):
        slice_scores[str(name)] = float(np.mean(score_array[slice_array == name]))
    if rng is None:
        rng = np.random.default_rng(17)
    bootstrap = []
    n = len(score_array)
    for _ in range(bootstrap_rounds):
        sample = rng.integers(0, n, size=n)
        bootstrap.append(float(np.mean(score_array[sample])))
    interval = np.quantile(bootstrap, [0.05, 0.95])
    mean_pass = mean_score >= threshold
    slice_pass = all(value >= slice_threshold for value in slice_scores.values())
    decision = bool(mean_pass and slice_pass)
    return {
        "mean": mean_score,
        "slice_scores": slice_scores,
        "interval": interval,
        "decision": decision,
    }

accuracy = 87 / 100
mobile = 42 / 50
desktop = 45 / 50
slice_gap = desktop - mobile
bootstrap_mean = np.mean([0.80, 0.85, 0.90])
win_rate = 34 / 60
assert abs(accuracy - 0.87) < 1e-12
assert abs(mobile - 0.84) < 1e-12
assert abs(desktop - 0.90) < 1e-12
assert abs(slice_gap - 0.06) < 1e-12
assert abs(bootstrap_mean - 0.85) < 1e-12
assert abs(win_rate - 0.5666666666666667) < 1e-12
print("accuracy", accuracy)
print("slice gap", slice_gap)
print("pairwise win rate", round(win_rate, 3))

The same method now becomes the single evaluation API. It returns the operational artifact and the topic metric so the D1 proof scales to D2–D5.

In [ ]:
print("Build-once assertions passed for 20.18")

## The dataset ladder

Family F17 uses an inline D1–D5 systems workload ladder. Each rung increases operational realism while keeping the computation CPU-only, seeded, and NumPy based.

In [ ]:
def make_eval_cases(n, rng, base=0.88, mobile_drop=0.0, drift=0.0):
    slices = rng.choice(["mobile", "desktop", "new_user", "power_user"], size=n, p=[0.35, 0.35, 0.15, 0.15])
    probability = np.full(n, base)
    probability[slices == "mobile"] = probability[slices == "mobile"] - mobile_drop
    probability = probability - drift * rng.random(n)
    probability = np.clip(probability, 0.05, 0.99)
    scores = (rng.random(n) < probability).astype(float)
    taxonomy = rng.choice(["retrieval", "format", "safety", "reasoning"], size=n)
    return {"scores": scores, "slices": slices, "taxonomy": taxonomy, "probability": probability}

def build_eval_ladder(seed=17):
    rng = np.random.default_rng(seed)
    ladder = []
    d1_scores = np.array([1, 1, 1, 0, 1, 1, 0, 1, 1, 1], dtype=float)
    d1_slices = np.array(["mobile", "desktop", "mobile", "desktop", "mobile", "desktop", "mobile", "desktop", "mobile", "desktop"])
    ladder.append({"name": "D1 ten hand cases", "cases": {"scores": d1_scores, "slices": d1_slices, "taxonomy": np.array(["fixed"] * 10)}, "threshold": 0.75, "load": 10})
    ladder.append({"name": "D2 stable 1k cases", "cases": make_eval_cases(1000, rng, 0.88), "threshold": 0.86, "load": 1000})
    ladder.append({"name": "D3 slice regression and drift", "cases": make_eval_cases(2500, rng, 0.89, 0.16, 0.05), "threshold": 0.86, "load": 2500})
    ladder.append({"name": "D4 case taxonomy log", "cases": make_eval_cases(6000, rng, 0.90, 0.06, 0.03), "threshold": 0.86, "load": 6000})
    ladder.append({"name": "D5 production release harness", "cases": make_eval_cases(14000, rng, 0.91, 0.18, 0.04), "threshold": 0.86, "load": 14000})
    return ladder

ladder = build_eval_ladder()
for rung in ladder:
    cases = rung["cases"]
    print(rung["name"], "n", len(cases["scores"]), "threshold", rung["threshold"], "slice sample", cases["slices"][:5].tolist())

## Run the same method across D1–D5

The metric is collected in the same way for every rung, so changes across the ladder reflect workload complexity rather than a different measurement.

In [ ]:
results = []
for rung in ladder:
    cases = rung["cases"]
    result = eval_harness(cases["scores"], cases["slices"], threshold=rung["threshold"], slice_threshold=0.82)
    oracle = np.mean(cases["probability"]) >= rung["threshold"] if "probability" in cases else result["mean"] >= rung["threshold"]
    decision_error = float(result["decision"] != bool(oracle))
    results.append({"name": rung["name"], "metric": decision_error, "artifact": result, "load": rung["load"], "cases": cases})

print("rung | decision error | mean | decision")
for item in results:
    print(item["name"], item["metric"], round(item["artifact"]["mean"], 3), item["artifact"]["decision"])

## Results visualization

The closing figure has operational panels for each rung plus a metric-vs-load curve.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for col, item in enumerate(results):
    slice_scores = item["artifact"]["slice_scores"]
    labels = list(slice_scores.keys())
    values = [slice_scores[label] for label in labels]
    axes[0, col].bar(labels, values, color="tab:purple")
    axes[0, col].set_ylim(0, 1)
    axes[0, col].set_title(item["name"].split()[0] + " slices")
    axes[0, col].tick_params(axis="x", rotation=45)
    cases = item["cases"]
    running = np.cumsum(cases["scores"]) / np.arange(1, len(cases["scores"]) + 1)
    axes[1, col].plot(running[:1000], color="tab:gray")
    axes[1, col].set_xlabel("case")
    axes[1, col].set_ylabel("running score")
fig.suptitle("Operational panels: slice metrics and running harness scores")
plt.tight_layout()

fig, ax = plt.subplots(figsize=(7, 4))
loads = [item["load"] for item in results]
metrics = [item["metric"] for item in results]
ax.plot(loads, metrics, marker="o")
ax.set_xlabel("case volume")
ax.set_ylabel("release decision error")
ax.set_title("Metric vs case volume")
plt.show()

## Pitfall on the hardest rung

Reproduce the lesson pitfall on D5, then apply the operational fix and compare the metric.

In [ ]:
d5 = results[-1]
mean_only = d5["artifact"]["mean"] >= 0.86
slice_aware = d5["artifact"]["decision"]
print("mean-only decision", bool(mean_only), "mean", round(d5["artifact"]["mean"], 3))
print("slice-aware decision", bool(slice_aware))
for name, value in d5["artifact"]["slice_scores"].items():
    print(name, round(value, 3))

## Evaluate it + Practice

- Metric: release decision error rate, compared with a mean-only baseline.
- Sanity check: 87/100 is 0.87 and the mobile-desktop gap is 0.06.
- Ablation: remove slice thresholds and verify a hidden mobile regression can ship.
- Failure signals: drifting case taxonomy, wide bootstrap intervals, and missing threshold decisions.

Practice prompts:
1. Change one workload knob on D3 and predict whether the metric should improve or degrade.
2. Add one slice or route to D4 and explain which operational panel should change.
3. Tighten the D5 budget or threshold and rerun only after saving your own copy.